# 04 Baseline: random split

Goal: train the same model with the same settings on a random split, as in the original project, to measure how much the split strategy affects the score.

Differences from notebook 03: the split file, the name of the saved model, and 5 epochs instead of 10.

In [3]:
import csv
from pathlib import Path
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from torch import nn
from torchvision import models
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
data_dir = Path("../data/Original CT Scans")
split_file = Path("../splits/random_split.csv")

In [4]:
train_items, val_items, test_items = [], [], []

with open(split_file, newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        item = (data_dir / row["path"], row["label"])
        if row["split"] == "train":
            train_items.append(item)
        elif row["split"] == "val":
            val_items.append(item)
        elif row["split"] == "test":
            test_items.append(item)

print(len(train_items), len(val_items), len(test_items))
print(train_items[0])
print(train_items[0][0].exists())

13778 2951 2956
(WindowsPath('../data/Original CT Scans/nCT/nCT1715.jpg'), 'nCT')
True


In [5]:
class_to_idx = {"nCT": 0, "NiCT": 1, "pCT": 2}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class CTDataset(Dataset):
    def __init__(self, items, transform):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        path, label = self.items[index]
        image = Image.open(path).convert("RGB")
        image = self.transform(image)
        return image, class_to_idx[label]

In [6]:
train_dataset = CTDataset(train_items, transform)
print(len(train_dataset))

image, label = train_dataset[0]
print(image.shape, label)
print(image.min(), image.max())

13778
torch.Size([3, 224, 224]) 0
tensor(-1.2617) tensor(2.6400)


In [7]:
val_dataset = CTDataset(val_items, transform)
test_dataset = CTDataset(test_items, transform)
print(len(val_dataset), len(test_dataset))


2951 2956


In [8]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(len(train_loader), len(val_loader), len(test_loader))

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)
print(labels)

431 93 93
torch.Size([32, 3, 224, 224]) torch.Size([32])
tensor([2, 1, 1, 0, 0, 0, 2, 1, 1, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        2, 0, 0, 2, 0, 1, 0, 0])


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
print(model.classifier)

cuda
Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=1000, bias=True)
)


In [10]:
model.classifier[3] = nn.Linear(1280, 3)
model = model.to(device)

print(sum(p.numel() for p in model.parameters()))

4205875


In [11]:
model.eval()
with torch.no_grad():
    outputs = model(images.to(device))

print(outputs.shape)
print(torch.softmax(outputs[0], dim=0))

torch.Size([32, 3])
tensor([0.3811, 0.2959, 0.3230], device='cuda:0')


In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

images_gpu = images.to(device)
labels_gpu = labels.to(device)

model.train()
for step in range(20):
    optimizer.zero_grad()
    outputs = model(images_gpu)
    loss = criterion(outputs, labels_gpu)
    loss.backward()
    optimizer.step()
    print(step, round(loss.item(), 4))

0 1.1059
1 0.951
2 0.8054
3 0.6857
4 0.5953
5 0.4837
6 0.4296
7 0.3493
8 0.2959
9 0.2529
10 0.2071
11 0.1768
12 0.1407
13 0.1236
14 0.0989
15 0.0809
16 0.0647
17 0.0563
18 0.0407
19 0.035


In [13]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [14]:
def evaluate(model,loader):
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * len(labels)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

In [15]:
torch.manual_seed(42)

model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
model.classifier[3] = nn.Linear(1280, 3)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

num_epochs = 5
best_val_loss = float("inf")
history = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)
    history.append((train_loss, train_acc, val_loss, val_acc))
    print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, acc {train_acc:.2%} | val loss {val_loss:.4f}, acc {val_acc:.2%}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), output_dir / "best_model_random.pth")
        print("  Saved best model")

Epoch 1: train loss 0.0922, acc 96.76% | val loss 0.0488, acc 98.34%
  Saved best model
Epoch 2: train loss 0.0197, acc 99.30% | val loss 0.0188, acc 99.36%
  Saved best model
Epoch 3: train loss 0.0111, acc 99.63% | val loss 0.0207, acc 99.29%
Epoch 4: train loss 0.0100, acc 99.67% | val loss 0.0193, acc 99.42%
Epoch 5: train loss 0.0099, acc 99.70% | val loss 0.0204, acc 99.19%


In [16]:
train_loss_eval, train_acc_eval = evaluate(model, train_loader)
print(f"Train in eval mode: loss {train_loss_eval:.4f}, acc {train_acc_eval:.2%}")

Train in eval mode: loss 0.0110, acc 99.68%


In [17]:
def predict(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(device))
            all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.tolist())
    return all_labels, all_preds

y_true, y_pred = predict(model, val_loader)
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["nCT", "NiCT", "pCT"]))

[[1480   16    0]
 [   1  853    1]
 [   0    6  594]]
              precision    recall  f1-score   support

         nCT       1.00      0.99      0.99      1496
        NiCT       0.97      1.00      0.99       855
         pCT       1.00      0.99      0.99       600

    accuracy                           0.99      2951
   macro avg       0.99      0.99      0.99      2951
weighted avg       0.99      0.99      0.99      2951



## Comparison with the block split

At the best epoch (epoch 2 in both runs):

- Random split: validation accuracy 99.4%, loss 0.019
- Block split: validation accuracy 63.2%, loss 2.07

Normal slices predicted as positive: 0 with the random split, 711 with the block split.

With a random split, neighbouring slices from the same scan end up in both train and validation, so the model can recognise scans it has already seen instead of the disease. This reproduces the 99% of the original project and suggests that most of that score comes from this leakage.

Limitations:

- One run per split, with one seed.
- The block split does not guarantee that patients are separated, and blocks may also differ in other ways, such as collection period or hospital.
- The test set has not been used yet.